In [1]:
import pandas as pd

pd.set_option("display.max_columns", 100)

games = pd.read_csv("C:\\Users\\Nico\\Documents\\The Power MBA\\Proyecto Final\\cs2-pro-scene-analysis\\data\\raw\\cs2_all_tiers_games.csv")
df = games.copy()

df = df.rename(columns={"bestOf": "best_of"})
df["datetime"] = pd.to_datetime(df["datetime"])
for col in ["tournament", "team1", "team2", "map_name"]:
    df[col] = df[col].str.strip()

In [2]:

import pandas as pd

pd.set_option("display.max_columns", 100)

games = pd.read_csv("C:\\Users\\Nico\\Documents\\The Power MBA\\Proyecto Final\\cs2-pro-scene-analysis\\data\\raw\\cs2_all_tiers_games.csv")
df = games.copy()

df = df.rename(columns={"bestOf": "best_of"})
df["datetime"] = pd.to_datetime(df["datetime"])
for col in ["tournament", "team1", "team2", "map_name"]:
    df[col] = df[col].str.strip()

In [3]:
tiers = []
for t in (1, 2, 3):
    ids = pd.read_csv(f"C:\\Users\\Nico\\Documents\\The Power MBA\\Proyecto Final\\cs2-pro-scene-analysis\\data\\raw\\cs2_tier{t}_games.csv", usecols=["match_id"]).drop_duplicates()
    ids["tier"] = t
    tiers.append(ids)
tiers = pd.concat(tiers)

print("match_id en más de un tier:", tiers["match_id"].duplicated().sum())
df = df.merge(tiers, on="match_id", how="left")
print("sin tier:", df["tier"].isna().sum(), df.shape)

match_id en más de un tier: 0
sin tier: 0 (23888, 99)


In [4]:
totals = df[df["is_total"]].copy()
maps = df[~df["is_total"]].copy()
print(totals.shape, maps.shape)

print("duplicados match_id en totales:", totals["match_id"].duplicated().sum())
totals = totals.drop_duplicates(subset="match_id", keep="first")
print(totals.shape)

(11398, 99) (12490, 99)
duplicados match_id en totales: 4
(11394, 99)


In [5]:
print(maps[["map_name", "score1_game", "score2_game", "best_of"]].isna().sum())

maps = maps.dropna(subset=["score1_game", "score2_game"]).copy()
maps["map_name"] = maps["map_name"].fillna("Desconocido")
print(maps.shape)

map_name       90
score1_game    35
score2_game    35
best_of        21
dtype: int64
(12455, 99)


In [6]:
incompletas = (maps["score1_game"] < 13) & (maps["score2_game"] < 13)
print("incompletas:", incompletas.sum())
maps = maps[~incompletas].copy()
print(maps.shape)

incompletas: 11
(12444, 99)


In [7]:
maps["team1_win_map"] = (maps["score1_game"] > maps["score2_game"]).astype(int)
print("coincide con team1_win:", (maps["team1_win_map"] == maps["team1_win"]).mean())

maps["total_rounds"] = maps["score1_game"] + maps["score2_game"]
maps["year"] = maps["datetime"].dt.year
maps["month"] = maps["datetime"].dt.month

coincide con team1_win: 0.9604628736740598


In [8]:
int_cols = ["game_id", "map_id", "score1_match", "score2_match", "best_of",
            "score1_game", "score2_game", "games_played", "team1_win"]
for col in int_cols:
    maps[col] = maps[col].astype("Int64")

print(maps.shape, "| partidas:", maps["match_id"].nunique())
print(maps["tier"].value_counts().to_dict())
print(maps["year"].value_counts().sort_index().to_dict())
print(maps["total_rounds"].describe())

(12444, 103) | partidas: 5997
{1: 7700, 2: 4253, 3: 491}
{2023: 683, 2024: 4458, 2025: 3693, 2026: 3610}
count    12444.000000
mean        21.706766
std          4.838590
min         13.000000
25%         19.000000
50%         21.000000
75%         23.000000
max         71.000000
Name: total_rounds, dtype: float64


In [9]:
cols_clave = ["team1", "team2", "score1_match", "score2_match", "team1_win"]
totals_malos = totals[totals[cols_clave].isna().any(axis=1)]
print("totales con datos clave vacíos:", len(totals_malos))
totals = totals.drop(totals_malos.index)
print(totals.shape)

maps.to_csv("C:\\Users\\Nico\\Documents\\The Power MBA\\Proyecto Final\\cs2-pro-scene-analysis\\data/processed/games_maps_clean.csv", index=False)
totals.to_csv("C:\\Users\\Nico\\Documents\\The Power MBA\\Proyecto Final\\cs2-pro-scene-analysis\\data/processed/games_totals_clean.csv", index=False)

totales con datos clave vacíos: 2
(11392, 99)
